# 🔍 Retrieval-Augmented Generation (RAG)

This notebook demonstrates:
- Loading vector stores
- Single-paper Q&A with LangChain
- Multi-paper synthesis
- Hybrid retrieval (semantic + keyword)
- Custom RAG chains

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

from embeddings import EmbeddingManager
from retrieval import RAGRetrieval, HybridRetriever
import json
import os

# Set API keys if not already set
# os.environ['GROQ_API_KEY'] = 'your-groq-key-here'
# os.environ['GOOGLE_API_KEY'] = 'your-google-key-here'

print("✓ Imports successful")

## 1. Load Vector Store

Load the previously created vector database.

In [ ]:
# Check for API keys (Groq for LLM, Google for embeddings)
if not os.getenv('GROQ_API_KEY') or not os.getenv('GOOGLE_API_KEY'):
    print("⚠️ Warning: Missing API keys!")
    print("This notebook requires GROQ_API_KEY and GOOGLE_API_KEY")
    print("Get Groq key at: https://console.groq.com")
    print("Get Google key at: https://makersuite.google.com/app/apikey")
    USE_MOCK = True
else:
    USE_MOCK = False
    
    # Initialize embedding manager with Google
    embedding_manager = EmbeddingManager(
        provider="google",
        persist_directory="../chroma_db"
    )
    
    # Load existing vector store
    try:
        vectorstore = embedding_manager.load_vectorstore(collection_name="sci_papers")
        print("✓ Vector store loaded (Google Gemini embeddings)")
    except Exception as e:
        print(f"⚠️ Could not load vector store: {e}")
        print("Please run 02_chunking_embeddings.ipynb first")
        USE_MOCK = True

## 2. Initialize RAG System

Set up the retrieval-augmented generation pipeline.

In [ ]:
if not USE_MOCK:
    # Initialize RAG retrieval with Groq (fast & free)
    rag = RAGRetrieval(
        vectorstore=vectorstore,
        llm_provider="groq",
        temperature=0.0  # Deterministic responses
    )
    
    print("✓ RAG system initialized")
    print(f"  LLM Provider: {provider}")
    print(f"  Temperature: 0.0")
else:
    print("⚠️ Using mock mode - skipping RAG initialization")

## 3. Single-Paper Q&A

Query a specific paper with source attribution.

In [ ]:
if not USE_MOCK:
    # Load paper metadata
    with open('../data/processed_papers.json', 'r') as f:
        papers = json.load(f)
    
    # Show available papers
    print("Available Papers:")
    for i, paper in enumerate(papers, 1):
        print(f"{i}. [{paper['paper_id']}] {paper['metadata']['title']}")
    
    # Query first paper
    paper_id = papers[0]['paper_id']
    question = "What are the main findings of this research?"
    
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"Target Paper: {papers[0]['metadata']['title'][:50]}...")
    print(f"{'='*60}\n")
    
    # Execute query
    result = rag.query_single_paper(
        question=question,
        paper_id=paper_id,
        k=5
    )
    
    # Display answer
    print("ANSWER:")
    print(result['answer'])
    
    # Display sources
    print(f"\n\nSOURCES ({len(result['source_documents'])} chunks):")
    for i, doc in enumerate(result['source_documents'], 1):
        print(f"\n  [{i}] Section: {doc.metadata.get('section', 'Unknown')}")
        print(f"      {doc.page_content[:150]}...")
else:
    print("⚠️ Skipped: No API key configured")
    print("\nExample output:")
    print("ANSWER: Based on the paper's results section, the main findings include...")

## 4. Multi-Paper Synthesis

Synthesize information across multiple papers.

In [ ]:
if not USE_MOCK:
    # Multi-paper question
    question = "What methodologies are commonly used across these papers?"
    
    print(f"{'='*60}")
    print(f"Multi-Paper Question: {question}")
    print(f"{'='*60}\n")
    
    # Execute query
    result = rag.query_multi_paper(
        question=question,
        k=10  # Retrieve more chunks for synthesis
    )
    
    # Display synthesized answer
    print("SYNTHESIZED ANSWER:")
    print(result['answer'])
    
    # Show which papers were cited
    print(f"\n\nPAPERS REFERENCED ({len(result['papers_cited'])}):")
    for paper_id, title in result['papers_cited'].items():
        print(f"  • {title}")
    
    # Optional: Show source distribution
    paper_counts = {}
    for doc in result['source_documents']:
        pid = doc.metadata.get('paper_id', 'Unknown')
        paper_counts[pid] = paper_counts.get(pid, 0) + 1
    
    print("\n\nSOURCE DISTRIBUTION:")
    for pid, count in paper_counts.items():
        print(f"  {pid}: {count} chunks")
else:
    print("⚠️ Skipped: No API key configured")
    print("\nExample output:")
    print("SYNTHESIZED ANSWER: Across the analyzed papers, several methodologies emerge...")

## 5. Hybrid Retrieval

Combine semantic (vector) and keyword (BM25) search.

In [ ]:
if not USE_MOCK:
    # Load chunks for BM25
    with open('../data/chunks.json', 'r') as f:
        chunks = json.load(f)
    
    documents = embedding_manager.chunks_to_documents(chunks)
    
    print(f"✓ Loaded {len(documents)} documents for hybrid retrieval")
    
    # Test hybrid search
    question = "What are the limitations of this approach?"
    
    print(f"\n{'='*60}")
    print(f"Hybrid Search Query: {question}")
    print(f"{'='*60}\n")
    
    # Execute hybrid search
    result = rag.hybrid_search_query(
        question=question,
        documents=documents,
        k=5,
        alpha=0.5  # Equal weight to semantic and keyword
    )
    
    # Display answer
    print("ANSWER (using hybrid retrieval):")
    print(result['answer'])
    
    # Display sources
    print(f"\n\nTOP SOURCES:")
    for i, doc in enumerate(result['source_documents'], 1):
        print(f"\n  [{i}] {doc.metadata.get('paper_title', 'Unknown')[:40]}...")
        print(f"      Section: {doc.metadata.get('section', 'Unknown')}")
        print(f"      {doc.page_content[:120]}...")
else:
    print("⚠️ Skipped: No API key configured")

## 6. Semantic vs Keyword Comparison

Compare pure semantic and pure keyword retrieval.

In [ ]:
if not USE_MOCK:
    query = "neural network architecture"
    
    print(f"Query: '{query}'\n")
    
    # Pure semantic (alpha=1.0)
    semantic_result = rag.hybrid_search_query(
        question=query,
        documents=documents,
        k=3,
        alpha=1.0  # 100% semantic
    )
    
    # Pure keyword (alpha=0.0)
    keyword_result = rag.hybrid_search_query(
        question=query,
        documents=documents,
        k=3,
        alpha=0.0  # 100% keyword (BM25)
    )
    
    # Display comparison
    print("🔮 SEMANTIC SEARCH (Vector-based):")
    for i, doc in enumerate(semantic_result['source_documents'], 1):
        print(f"  {i}. {doc.page_content[:80]}...")
    
    print("\n\n🔑 KEYWORD SEARCH (BM25):")
    for i, doc in enumerate(keyword_result['source_documents'], 1):
        print(f"  {i}. {doc.page_content[:80]}...")
else:
    print("⚠️ Skipped: No API key configured")

## 7. Custom Retrieval with Filters

Use metadata filters for targeted retrieval.

In [ ]:
if not USE_MOCK:
    # Get unique sections
    sections = set(chunk.get('section') for chunk in chunks)
    print(f"Available sections: {sections}\n")
    
    # Query specific section
    target_section = "methods"  # Change as needed
    
    results = embedding_manager.similarity_search(
        query="What approach was used?",
        k=3,
        filter_dict={"section": target_section}
    )
    
    print(f"Results from '{target_section}' section only:\n")
    for i, doc in enumerate(results, 1):
        print(f"  [{i}] {doc.metadata.get('paper_title', 'Unknown')[:40]}...")
        print(f"      {doc.page_content[:100]}...\n")
else:
    print("⚠️ Skipped: No API key configured")

## 8. Retrieval Quality Analysis

Analyze the quality and relevance of retrieved chunks.

In [ ]:
if not USE_MOCK:
    # Test multiple queries
    test_queries = [
        "What is the main contribution?",
        "What dataset was used?",
        "What are the experimental results?",
        "What are future research directions?"
    ]
    
    print("Retrieval Quality Test:\n")
    
    for query in test_queries:
        results = embedding_manager.similarity_search_with_score(query, k=3)
        
        print(f"Query: '{query}'")
        print(f"  Top score: {results[0][1]:.4f}")
        print(f"  Avg score: {sum(r[1] for r in results) / len(results):.4f}")
        print(f"  Sections: {[r[0].metadata.get('section', '?') for r in results]}")
        print()
else:
    print("⚠️ Skipped: No API key configured")

## 9. Interactive Q&A

Try your own questions! (modify and re-run this cell)

In [ ]:
if not USE_MOCK:
    # ===== MODIFY THIS SECTION =====
    YOUR_QUESTION = "What are the key findings?"
    USE_SINGLE_PAPER = True  # Set to False for multi-paper
    TARGET_PAPER_ID = papers[0]['paper_id']  # Only used if USE_SINGLE_PAPER=True
    # ================================
    
    if USE_SINGLE_PAPER:
        result = rag.query_single_paper(
            question=YOUR_QUESTION,
            paper_id=TARGET_PAPER_ID,
            k=5
        )
    else:
        result = rag.query_multi_paper(
            question=YOUR_QUESTION,
            k=10
        )
    
    print(f"Q: {YOUR_QUESTION}\n")
    print(f"A: {result['answer']}")
else:
    print("⚠️ Skipped: No API keys configured")
    print("\nTo use interactive Q&A:")
    print("1. Set GROQ_API_KEY and GOOGLE_API_KEY")
    print("2. Run 02_chunking_embeddings.ipynb to create vector store")
    print("3. Restart this notebook")

## Summary

✅ Vector store loaded successfully  
✅ Single-paper Q&A demonstrated  
✅ Multi-paper synthesis tested  
✅ Hybrid retrieval (semantic + keyword) implemented  
✅ Metadata filtering explored  

⏭️ Next: Move to `04_analysis_layers.ipynb` for advanced analysis